In [32]:
#%pip install -q --no-deps \
#  langgraph==0.2.45 \
#  langchain==0.1.20 \
#  langchain-core==0.1.53 \
#  langchain-google-genai==1.0.10


In [33]:
import langgraph
import langchain
import langchain_core

print("LangGraph module loaded from:", langgraph.__file__)
print("LangChain version:", langchain.__version__)
print("LangChain Core version:", langchain_core.__version__)


LangGraph module loaded from: None
LangChain version: 1.2.0
LangChain Core version: 1.2.3


In [34]:
import os
from google.colab import userdata

GOOGLE_API_KEY = userdata.get("GEMINI_API_KEY")
if not GOOGLE_API_KEY:
    raise ValueError("Missing GEMINI_API_KEY in Colab Secrets")

os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY


In [35]:
from typing import Annotated, Literal
from typing_extensions import TypedDict

from langgraph.graph.message import add_messages


class OrderState(TypedDict):
    """State representing the customer's order conversation."""
    messages: Annotated[list, add_messages]
    order: list[str]
    finished: bool


BARISTABOT_SYSINT = (
    "system",
    "You are a BaristaBot, an interactive cafe ordering system. "
    "You only discuss menu items. "
    "Use tools to manage the order. "
    "Always confirm the order before placing it. "
    "Once the order is placed, thank the customer and say goodbye."
)

WELCOME_MSG = "Welcome to the BaristaBot cafe. Type `q` to quit. How may I serve you today?"


In [36]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(
    model="models/gemini-flash-lite-latest",
    temperature=0.3,
)


In [37]:
from typing import Annotated, Literal
from typing_extensions import TypedDict

from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages


In [38]:
class OrderState(TypedDict):
    """State representing the customer's order conversation."""
    messages: Annotated[list, add_messages]
    order: list[str]
    finished: bool


BARISTABOT_SYSINT = (
    "system",
    "You are a BaristaBot, an interactive cafe ordering system. "
    "You only discuss menu items. "
    "Use tools to manage the order. "
    "Always confirm the order before placing it. "
    "Once the order is placed, thank the customer and say goodbye."
)

WELCOME_MSG = "Welcome to the BaristaBot cafe. Type `q` to quit. How may I serve you today?"


In [39]:
from langchain_core.messages.ai import AIMessage


def chatbot_with_welcome(state: OrderState) -> OrderState:
    """Chatbot node that starts the conversation and handles LLM replies."""
    if state["messages"]:
        ai_msg = llm.invoke([BARISTABOT_SYSINT] + state["messages"])
    else:
        ai_msg = AIMessage(content=WELCOME_MSG)

    return state | {"messages": [ai_msg]}


In [40]:
def human_node(state: OrderState) -> OrderState:
    """Display last model message and receive user input."""
    last_msg = state["messages"][-1]
    print("BaristaBot:", last_msg.content)

    user_input = input("User: ")

    if user_input.lower() in {"q", "quit", "exit", "goodbye"}:
        state["finished"] = True

    return state | {"messages": [("user", user_input)]}


In [41]:
def maybe_exit_human_node(state: OrderState) -> Literal["chatbot", "__end__"]:
    if state.get("finished", False):
        return END
    return "chatbot"


In [42]:
from langchain_core.tools import tool


@tool
def get_menu() -> str:
    """Provide the latest up-to-date menu."""
    return """
MENU:
Coffee Drinks:
- Espresso
- Americano
- Cold Brew

Coffee Drinks with Milk:
- Latte
- Cappuccino
- Flat White

Tea Drinks:
- English Breakfast Tea
- Green Tea
- Earl Grey

Tea Drinks with Milk:
- Chai Latte
- Matcha Latte

Modifiers:
Milk: Whole, Oat, Almond
Shots: Single, Double
Caffeine: Regular, Decaf

Soy milk is out of stock today.
"""


In [46]:
import langgraph
print("LangGraph OK")


LangGraph OK


In [49]:
def tools_node(state: OrderState) -> OrderState:
    """Custom tools node replacing ToolNode (Colab compatible)."""
    msg = state["messages"][-1]
    outbound_msgs = []

    for call in getattr(msg, "tool_calls", []):
        if call["name"] == "get_menu":
            result = get_menu()
        else:
            raise ValueError(f"Unknown tool: {call['name']}")

        outbound_msgs.append(
            {
                "role": "tool",
                "tool_call_id": call["id"],
                "content": result,
            }
        )

    return {"messages": outbound_msgs}


In [50]:
def maybe_route_to_tools(state: OrderState):
    msg = state["messages"][-1]
    if hasattr(msg, "tool_calls") and msg.tool_calls:
        return "tools"
    return "human"


In [52]:
graph_builder = StateGraph(OrderState)

graph_builder.add_node("chatbot", chatbot_with_welcome)
graph_builder.add_node("human", human_node)
graph_builder.add_node("tools", tools_node)

graph_builder.add_conditional_edges("chatbot", maybe_route_to_tools)
graph_builder.add_conditional_edges("human", maybe_exit_human_node)

graph_builder.add_edge("tools", "chatbot")
graph_builder.add_edge(START, "chatbot")

graph_with_menu = graph_builder.compile()


In [54]:
config = {"recursion_limit": 50}

state = graph_with_menu.invoke(
    {
        "messages": [],
        "order": [],
        "finished": False,
    },
    config=config,
)


BaristaBot: Welcome to the BaristaBot cafe. Type `q` to quit. How may I serve you today?
User: show me the menu
BaristaBot: I can show you our menu. What would you like to see: **Drinks** or **Food**?
User: drinks
BaristaBot: Here are our drink options:

| Item | Price |
| :--- | :--- |
| Espresso | \$2.50 |
| Americano | \$3.00 |
| Latte | \$4.50 |
| Cappuccino | \$4.50 |
| Mocha | \$5.00 |
| Iced Coffee | \$3.50 |
| Hot Tea | \$3.00 |

What can I get started for you?
User: one espresso and one mocha please
BaristaBot: Got it. I have one Espresso and one Mocha.

Would you like any customizations for those, or are you ready to confirm your order?
User: can i see the food also ? 
BaristaBot: Certainly. Here is our food menu:

| Item | Price |
| :--- | :--- |
| Croissant | \$3.50 |
| Muffin | \$3.00 |
| Scone | \$3.25 |
| Bagel | \$2.75 |

Would you like to add anything from the food menu to your order of one Espresso and one Mocha?
User: give me 2 croisants and one bagel please
BaristaB